In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')  

# Funciones Utiles

In [ ]:
# Eventos por event_name con descripcion
descripciones = {
    'Pass': 'Pase a un compañero',
    'BallTouch': 'Toque de balón sin pase',
    'BallRecovery': 'Recuperación del balón',
    'Clearance': 'Despeje',
    'BlockedPass': 'Pase bloqueado por rival',
    'Foul': 'Falta cometida',
    'Aerial': 'Duelo aéreo',
    'Claim': 'Arquero atrapa el balón',
    'KeeperPickup': 'Arquero recoge con las manos',
    'TakeOn': 'Intento de gambeta',
    'Tackle': 'Entrada/tackle al rival',
    'OffsideGiven': 'Offside cobrado',
    'Dispossessed': 'Jugador pierde el balón',
    'OffsidePass': 'Pase en posición de offside',
    'OffsideProvoked': 'Offside provocado al rival',
    'SavedShot': 'Tiro al arco atajado',
    'Save': 'Atajada del arquero',
    'ShieldBallOpp': 'Jugador protege el balón del rival',
    'MissedShots': 'Tiro desviado o al palo',
    'Card': 'Tarjeta amarilla o roja',
    'Punch': 'Arquero rechaza de puño',
    'Goal': 'Gol',
    'ChanceMissed': 'Chance de gol desperdiciada',
    'KeeperSweeper': 'Arquero sale a cortar',
    'Interception': 'Intercepción del balón',
    'SubstitutionOff': 'Jugador sale del partido',
    'SubstitutionOn': 'Jugador entra al partido',
    'FormationSet': 'Formación inicial registrada',
    'FormationChange': 'Cambio de formación',
    'Start': 'Inicio de período',
    'End': 'Fin de período',
    'Error': 'Error del jugador que genera chance rival',
    'ShotOnPost': 'Tiro que pega en el palo',
    'PenaltyFaced': 'Penal que enfrenta el arquero',
    'Smother': 'Arquero cierra el ángulo y tapa',
    'GoodSkill': 'Habilidad técnica destacada',
    'CrossNotClaimed': 'Centro no atrapado por el arquero',
}

def eventos_por_tipo(df):
    resultado = (
        df.groupby('event_name')
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )
    resultado['descripcion'] = resultado['event_name'].map(descripciones)
    return resultado

In [ ]:
def eventos_por_tipo(df):
    total = df.groupby('event_name').size().reset_index(name='count')
    exitosos = df[df['outcome_type'] == 'Successful'].groupby('event_name').size().reset_index(name='exitosos')
    
    resultado = total.merge(exitosos, on='event_name', how='left')
    resultado['exitosos'] = resultado['exitosos'].fillna(0)
    resultado['pct_exitoso'] = (resultado['exitosos'] / resultado['count'] * 100).round(1)
    resultado['descripcion'] = resultado['event_name'].map(descripciones)
    
    return resultado.drop(columns='exitosos').sort_values('count', ascending=False)

In [ ]:
from IPython.display import display, HTML

def _add_col_to_serie(serie, subset, col):
    nans = subset[col].isna().sum()
    serie[f'{col}_nan'] = nans

    TOP_N = {
    'jugador': 3,
    'previous_event': 3,
    'next_event_posesion': 3,
    }

    if col == 'period_id':
        vc = subset[col].value_counts()
        for period in [1, 2]:
            serie[f'period_id_{period}_count'] = vc.get(period, 0)
    else:
        n = TOP_N.get(col, 1)
        top = subset[col].value_counts().head(n)
        for i, (val, cnt) in enumerate(top.items(), 1):
            serie[f'{col}_top{i}']       = val
            serie[f'{col}_top{i}_count'] = cnt

def _cat_block(subset, cols):
    html = ""
    for col in cols:
        nans = subset[col].isna().sum()
        top  = subset[col].value_counts().head(5)
        top_df = top.reset_index()
        top_df.columns = [col, 'count']
        html += f"<b>{col}</b> — NaN: {nans:,}<br>" + top_df.to_html(index=False) + "<br>"
    return html

def analisis_evento(df, event_name):
    subset = df[df['event_name'] == event_name]
    n = len(subset)
    serie = {}

    # --- Numéricas ---
    col_num_pres = [c for c in Col_num if c in subset.columns]
    num_html = ""
    if col_num_pres:
        stats = subset[col_num_pres].agg(['mean', 'std']).T.round(3)
        stats.columns = ['Media', 'Desvio std']
        for col in col_num_pres:
            serie[f'{col}_mean'] = stats.loc[col, 'Media']
            serie[f'{col}_std']  = stats.loc[col, 'Desvio std']
        num_html = stats.to_html()

    # --- Outcome (solo outcome_type) ---
    outcome_html = ""
    if 'outcome_type' in subset.columns:
        nans = subset['outcome_type'].isna().sum()
        vc   = subset['outcome_type'].value_counts()
        out  = pd.DataFrame({'count': vc, '%': (vc / n * 100).round(1)}).head(5)
        for val, row in out.iterrows():
            serie[f'outcome_type_{val}_pct'] = row['%']
        outcome_html = f"<br><b>outcome_type</b> — NaN: {nans:,}<br>" + out.to_html()

    # --- Categóricas divididas en 2 ---
    col_count_pres = [c for c in Col_Count if c in subset.columns]
    for col in col_count_pres:
        _add_col_to_serie(serie, subset, col)

    mid = len(col_count_pres) // 2 + len(col_count_pres) % 2
    cat_html_1 = _cat_block(subset, col_count_pres[:mid])
    cat_html_2 = _cat_block(subset, col_count_pres[mid:])

    # --- Layout HTML ---
    html = f"""
    <h3>{event_name} &mdash; {n:,} eventos</h3>
    <table width="100%" style="border-collapse:collapse"><tr>
        <td valign="top" width="25%" style="padding-right:20px">
            <b>Numéricas</b><br>{num_html}{outcome_html}
        </td>
        <td valign="top" width="37%" style="padding-right:20px">
            <b>Categóricas (1/2)</b><br>{cat_html_1}
        </td>
        <td valign="top" width="38%">
            <b>Categóricas (2/2)</b><br>{cat_html_2}
        </td>
    </tr></table>
    """
    display(HTML(html))

    return pd.Series(serie, name=event_name)

In [ ]:
descripciones_posicion = {
    'GK':  'Arquero',
    'DC':  'Defensor central',
    'DL':  'Lateral izquierdo',
    'DR':  'Lateral derecho',
    'DML': 'Mediocampista defensivo izquierdo',
    'DMR': 'Mediocampista defensivo derecho',
    'DMC': 'Mediocampista defensivo central',
    'ML':  'Mediocampista izquierdo',
    'MR':  'Mediocampista derecho',
    'MC':  'Mediocampista central',
    'AML': 'Extremo izquierdo',
    'AMR': 'Extremo derecho',
    'AMC': 'Mediocampista ofensivo central',
    'FWL': 'Delantero izquierdo',
    'FWR': 'Delantero derecho',
    'FW':  'Delantero centro',
    'Sub': 'Suplente sin posicion definida',
}

def jugadores_por_posicion(df):
    resultado = (
        df.dropna(subset=['position'])
        .groupby('position')['jugador']
        .nunique()
        .reset_index(name='jugadores')
        .sort_values('jugadores', ascending=False)
    )
    resultado['descripcion'] = resultado['position'].map(descripciones_posicion)
    return resultado

In [ ]:
def dashboard_jugador(df, jugador):
    player_df = df[df['jugador'] == jugador]
    if len(player_df) == 0:
        display(HTML(f"<b>Jugador '{jugador}' no encontrado.</b>"))
        return

    partidos   = player_df['matchId'].nunique()
    equipo     = player_df['TeamName'].mode()[0]
    posiciones = player_df['position'].dropna().unique().tolist()

    def get_minutes(match_df):
        sub_on  = match_df[match_df['event_name'] == 'SubstitutionOn']['minute'].values
        sub_off = match_df[match_df['event_name'] == 'SubstitutionOff']['minute'].values
        start = sub_on[0]  if len(sub_on)  > 0 else 0
        end   = sub_off[0] if len(sub_off) > 0 else 90
        return max(0, end - start)

    minutos = int(player_df.groupby('matchId').apply(get_minutes).sum())
    per90   = minutos / 90 if minutos > 0 else 1

    goles  = len(player_df[player_df['event_name'] == 'Goal'])
    tiros  = int(player_df['isShot'].sum()) if 'isShot' in player_df.columns else 0

    pases_total    = len(player_df[player_df['event_name'] == 'Pass'])
    pases_exitosos = len(player_df[(player_df['event_name'] == 'Pass') & (player_df['outcome_type'] == 'Successful')])
    pct_pases      = f"{round(pases_exitosos / pases_total * 100, 1)}%" if pases_total > 0 else '—'

    amarillas = len(player_df[(player_df['event_name'] == 'Card') & (player_df['cardType'] == 'Yellow')])
    rojas     = len(player_df[(player_df['event_name'] == 'Card') & (player_df['cardType'].isin(['Red', 'SecondYellow']))])

    xG_90   = round(player_df['xG_corr'].sum()   / per90, 3) if 'xG_corr'   in player_df.columns else '—'
    xGoT_90 = round(player_df['xGoT_corr'].sum() / per90, 3) if 'xGoT_corr' in player_df.columns else '—'
    xG_tiro = round(player_df['xG_corr'].sum() / tiros, 3)   if tiros > 0 else '—'

    stats = [
        ('Partidos',  partidos,   '#48BF5F'),
        ('Minutos',   minutos,    '#48BF5F'),
        ('Goles',     goles,      '#48BF5F'),
        ('Pases',     pases_total,'#48BF5F'),
        ('% Pases',   pct_pases,  '#48BF5F'),
        ('Tiros',     tiros,      '#48BF5F'),
        ('xG/tiro',   xG_tiro,    '#48BF5F'),
        ('Amarillas', amarillas,  '#f1c40f'),
        ('Rojas',     rojas,      '#e74c3c'),
        ('xG/90',     xG_90,      '#48BF5F'),
        ('xGoT/90',   xGoT_90,    '#48BF5F'),
    ]

    cards_html = ''.join([
        f"""<div style="display:inline-block; background:#f5fff7; border-left:4px solid {color};
                        padding:10px 16px; border-radius:4px; margin:5px; min-width:90px; vertical-align:top">
                <div style="font-size:20px; font-weight:bold; color:#000">{value}</div>
                <div style="font-size:10px; color:#666; text-transform:uppercase; letter-spacing:1px; margin-top:2px">{label}</div>
            </div>"""
        for label, value, color in stats
    ])

    pos_badges = ''.join([
        f'<span style="background:rgba(72,191,95,0.2); color:#48BF5F; padding:2px 9px; border-radius:10px; font-size:11px; margin-right:5px">{p}</span>'
        for p in posiciones
    ])

    eventos_html = (
        player_df['event_name'].value_counts()
        .head(10)
        .reset_index()
        .rename(columns={'event_name': 'Evento', 'count': 'N'})
        .to_html(index=False)
    )

    html = f"""
    <style>td table {{ width: auto !important; }}</style>
    <div style="font-family:sans-serif; max-width:900px">
      <div style="background:#000; color:white; padding:16px 22px; border-radius:8px; margin-bottom:14px;
                  border-left:5px solid #48BF5F">
        <div style="font-size:22px; font-weight:bold; letter-spacing:0.5px">{jugador}</div>
        <div style="color:#aaa; font-size:13px; margin-top:4px">{equipo}</div>
        <div style="margin-top:8px">{pos_badges}</div>
      </div>
      <table width="100%" style="border-collapse:collapse">
        <tr>
          <td valign="top" width="62%" style="padding-right:24px; text-align:left">
            <div style="font-size:11px; color:#666; text-transform:uppercase; letter-spacing:1px; margin-bottom:8px">Stats</div>
            {cards_html}
          </td>
          <td valign="top" width="38%" style="text-align:left">
            <div style="font-size:11px; color:#666; text-transform:uppercase; letter-spacing:1px; margin-bottom:8px">Top 10 Eventos</div>
            {eventos_html}
          </td>
        </tr>
      </table>
    </div>
    """
    display(HTML(html))

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

def buscador_jugador(df):
    jugadores = sorted(df['jugador'].dropna().unique().tolist())

    titulo = widgets.HTML("""
        <div style="background:linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
                    color:white; padding:14px 22px; border-radius:8px;
                    font-family:sans-serif; margin-bottom:10px">
            <div style="font-size:17px; font-weight:bold; letter-spacing:0.5px">Buscador de Jugadores</div>
            <div style="color:#aaa; font-size:12px; margin-top:3px">
                Escribí al menos 2 letras y seleccioná un resultado
            </div>
        </div>
    """)

    search    = widgets.Text(placeholder='Ej: Sala, Haal, De Br...', layout=widgets.Layout(width='340px'))
    resultados = widgets.Output()
    dashboard  = widgets.Output()

    def on_type(change):
        query = change['new'].strip().lower()
        with resultados:
            clear_output(wait=True)
            if len(query) < 2:
                return
            matches = [j for j in jugadores if query in j.lower()][:10]
            if not matches:
                display(widgets.HTML("<i style='color:#999; font-size:12px'>Sin resultados</i>"))
                return
            botones = [
                widgets.Button(
                    description=j,
                    layout=widgets.Layout(width='340px', margin='2px 0'),
                    style=widgets.ButtonStyle(button_color='#f0f4ff', font_weight='normal')
                )
                for j in matches
            ]
            for b in botones:
                def on_click(btn):
                    with resultados:
                        clear_output()
                    search.value = btn.description
                    with dashboard:
                        clear_output(wait=True)
                        dashboard_jugador(df, btn.description)
                b.on_click(on_click)
            display(widgets.VBox(botones))

    search.observe(on_type, names='value')

    display(widgets.VBox(
        [titulo, search, resultados, dashboard],
        layout=widgets.Layout(padding='8px')
    ))

In [ ]:
def dashboard_partido(df, match_id):
    match_df = df[df['matchId'] == match_id]
    equipos  = match_df['TeamName'].unique()
    if len(equipos) < 2:
        display(HTML("<b>Datos insuficientes.</b>"))
        return

    eq_a, eq_b = equipos[0], equipos[1]
    fecha = match_df['fecha'].iloc[0] if 'fecha' in match_df.columns else '—'
    df_a  = match_df[match_df['TeamName'] == eq_a]
    df_b  = match_df[match_df['TeamName'] == eq_b]

    goles_a = df_a['goles_equipo'].max() if 'goles_equipo' in df_a.columns else '—'
    goles_b = df_b['goles_equipo'].max() if 'goles_equipo' in df_b.columns else '—'

    def team_stats(tdf):
        pases    = len(tdf[tdf['event_name'] == 'Pass'])
        pases_ok = len(tdf[(tdf['event_name'] == 'Pass') & (tdf['outcome_type'] == 'Successful')])
        tiros    = int(tdf['isShot'].sum()) if 'isShot' in tdf.columns else '—'
        xG       = round(tdf['xG_corr'].sum(), 2) if 'xG_corr' in tdf.columns else '—'
        return {
            'Eventos':      len(tdf),
            'Tipos únicos': tdf['event_name'].nunique(),
            '1er tiempo':   len(tdf[tdf['period_id'] == 1]) if 'period_id' in tdf.columns else '—',
            '2do tiempo':   len(tdf[tdf['period_id'] == 2]) if 'period_id' in tdf.columns else '—',
            'Pases':        pases,
            '% Pases':      f"{round(pases_ok/pases*100,1)}%" if pases > 0 else '—',
            'Tiros':        tiros,
            'xG':           xG,
        }

    sa, sb = team_stats(df_a), team_stats(df_b)

    colors = {
        'Eventos':'#4a6cf7','Tipos únicos':'#4a6cf7',
        '1er tiempo':'#7f8c8d','2do tiempo':'#7f8c8d',
        'Pases':'#3498db','% Pases':'#3498db',
        'Tiros':'#e67e22','xG':'#e67e22',
    }

    rows_html = ''.join([
        f"""<tr>
            <td style="text-align:right;padding:7px 12px;font-weight:bold;font-size:15px;color:#1a1a2e">{sa[k]}</td>
            <td style="text-align:center;padding:7px 10px;font-size:10px;color:#888;text-transform:uppercase;
                       letter-spacing:1px;border-left:3px solid {colors.get(k,'#ccc')};
                       border-right:3px solid {colors.get(k,'#ccc')}">{k}</td>
            <td style="text-align:left;padding:7px 12px;font-weight:bold;font-size:15px;color:#1a1a2e">{sb[k]}</td>
        </tr>"""
        for k in sa
    ])

    ev_a  = sorted(df_a['event_name'].unique())
    ev_b  = sorted(df_b['event_name'].unique())
    todos = sorted(set(ev_a) | set(ev_b))

    ev_rows = ''.join([
        f"""<tr>
            <td style="padding:3px 10px;font-size:12px;color:{'#1a1a2e' if e in ev_a else '#ccc'};text-align:right">
                {'● ' + e if e in ev_a else ''}
            </td>
            <td style="padding:3px 10px;font-size:12px;color:{'#1a1a2e' if e in ev_b else '#ccc'};text-align:left">
                {'● ' + e if e in ev_b else ''}
            </td>
        </tr>"""
        for e in todos
    ])

    html = f"""
    <style>td table {{ width: auto !important; }}</style>
    <div style="font-family:sans-serif; max-width:680px">
      <div style="background:linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
                  color:white; padding:16px 22px; border-radius:8px; margin-bottom:14px; text-align:center">
        <div style="color:#aaa; font-size:12px; margin-bottom:8px">{fecha}</div>
        <div style="font-size:22px; font-weight:bold; letter-spacing:1px">
          {eq_a} &nbsp; <span style="color:#4a6cf7">{goles_a}</span>
          &nbsp;–&nbsp;
          <span style="color:#4a6cf7">{goles_b}</span> &nbsp; {eq_b}
        </div>
      </div>
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;
                  font-size:13px;font-weight:bold;color:#555;padding:0 12px">
        <span>{eq_a}</span><span>{eq_b}</span>
      </div>
      <table width="100%" style="border-collapse:collapse;background:#f8faff;border-radius:6px">
        {rows_html}
      </table>
      <div style="margin-top:16px;font-size:11px;color:#999;font-weight:bold;
                  text-transform:uppercase;letter-spacing:1px;margin-bottom:6px">
        Eventos por equipo
      </div>
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;
                  font-size:11px;font-weight:bold;color:#555">
        <span>{eq_a}</span><span>{eq_b}</span>
      </div>
      <table width="100%" style="border-collapse:collapse;background:#f8faff;border-radius:6px">
        {ev_rows}
      </table>
    </div>
    """
    display(HTML(html))


def buscador_partido(df):
    equipos  = sorted(df['TeamName'].dropna().unique().tolist())
    selected = {'eq1': None, 'eq2': None}

    titulo = widgets.HTML("""
        <div style="background:linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
                    color:white; padding:14px 22px; border-radius:8px; font-family:sans-serif; margin-bottom:12px">
            <div style="font-size:17px; font-weight:bold">Buscador de Partidos</div>
            <div style="color:#aaa; font-size:12px; margin-top:3px">Seleccioná los dos equipos — elegí el partido por fecha</div>
        </div>
    """)

    dashboard_out = widgets.Output()

    def make_col(role, placeholder):
        search  = widgets.Text(placeholder=placeholder, layout=widgets.Layout(width='220px'))
        results = widgets.Output()
        label   = widgets.HTML("")

        def on_type(change):
            query = change['new'].strip().lower()
            with results:
                clear_output(wait=True)
                if len(query) < 2:
                    return
                matches = [e for e in equipos if query in e.lower()][:8]
                if not matches:
                    display(widgets.HTML("<i style='color:#999;font-size:12px'>Sin resultados</i>"))
                    return
                botones = [
                    widgets.Button(description=e,
                                   layout=widgets.Layout(width='220px', margin='2px 0'),
                                   style=widgets.ButtonStyle(button_color='#f0f4ff'))
                    for e in matches
                ]
                for b in botones:
                    def on_click(btn, r=role, sw=search, ro=results, lw=label):
                        selected[r] = btn.description
                        sw.value    = btn.description
                        lw.value    = f"<span style='color:#2ecc71;font-size:12px'>✓ {btn.description}</span>"
                        with ro:
                            clear_output()
                        if selected['eq1'] and selected['eq2']:
                            show_matches()
                    b.on_click(on_click)
                display(widgets.VBox(botones))

        search.observe(on_type, names='value')
        col = widgets.VBox([
            widgets.HTML(f"<div style='font-size:11px;color:#999;text-transform:uppercase;letter-spacing:1px;margin-bottom:4px'>{placeholder.replace('...','')}</div>"),
            search, label, results
        ])
        return col

    def show_matches():
        with dashboard_out:
            clear_output(wait=True)
            eq1, eq2 = selected['eq1'], selected['eq2']
            ids1 = set(df[df['TeamName'] == eq1]['matchId'].unique())
            ids2 = set(df[df['TeamName'] == eq2]['matchId'].unique())
            ids  = sorted(ids1 & ids2)

            if not ids:
                display(HTML(f"<b>No se encontró partido entre {eq1} y {eq2}.</b>"))
                return
            if len(ids) == 1:
                dashboard_partido(df, ids[0])
                return

            display(HTML("<div style='font-family:sans-serif;font-size:13px;font-weight:bold;margin-bottom:8px'>Seleccioná el partido:</div>"))
            for mid in ids:
                mdf   = df[df['matchId'] == mid]
                fecha = mdf['fecha'].iloc[0]
                g1    = mdf[mdf['TeamName'] == eq1]['goles_equipo'].max()
                g2    = mdf[mdf['TeamName'] == eq2]['goles_equipo'].max()
                label = f"{fecha}  —  {eq1} {g1} – {g2} {eq2}"
                b = widgets.Button(description=label, layout=widgets.Layout(width='480px', margin='3px 0'))
                def on_click(btn, m=mid):
                    with dashboard_out:
                        clear_output(wait=True)
                        dashboard_partido(df, m)
                b.on_click(on_click)
                display(b)

    col1     = make_col('eq1', 'Equipo 1...')
    col2     = make_col('eq2', 'Equipo 2...')
    vs_label = widgets.HTML("<div style='padding:28px 18px;font-size:18px;color:#999'>vs</div>")

    display(widgets.VBox([
        titulo,
        widgets.HBox([col1, vs_label, col2]),
        dashboard_out
    ], layout=widgets.Layout(padding='8px')))

In [ ]:
import ast

def build_qualifiers_df(df):
    def parse_row(q_str):
        if not isinstance(q_str, str) or q_str.strip() == '[]':
            return {}
        parsed = ast.literal_eval(q_str)
        return {q['type']['displayName']: q.get('value', True) for q in parsed}

    parsed = df['qualifiers'].apply(parse_row)
    result = pd.DataFrame(parsed.tolist(), index=df['id'])

    for col in result.columns:
        try:
            result[col] = pd.to_numeric(result[col])
        except (ValueError, TypeError):
            pass

    return result

# Prueba con muestra chica:
# sample = PremierLeague_23_24.sample(1000)
# qualifiers_df = build_qualifiers_df(sample)
# qualifiers_df

In [ ]:
from mplsoccer import Pitch
from matplotlib.colors import LogNorm

def heatmap_pases(df, bins=False, bins_size=(50, 32), filter_set_pieces=True, log_norm=True):
    passes = df.dropna(subset=['x', 'y', 'endX', 'endY'])

    if filter_set_pieces:
        corners = (passes['x'] > 99.5) | (passes['x'] < 0.5)
        corners &= (passes['y'] > 99.5) | (passes['y'] < 0.5)
        kickoff = passes['x'].between(49.5, 50.5) & passes['y'].between(49.5, 50.5)
        passes = passes[~(corners | kickoff)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, (xcol, ycol), title in zip(
        axes,
        [('x', 'y'), ('endX', 'endY')],
        ['Origen de pases', 'Destino de pases']
    ):
        pitch = Pitch(pitch_type='opta', pitch_color='grass', line_color='white')
        pitch.draw(ax=ax)
        if bins:
            bs = pitch.bin_statistic(passes[xcol], passes[ycol], bins=bins_size)
            norm = LogNorm() if log_norm else None
            pitch.heatmap(bs, ax=ax, cmap='hot', norm=norm)
        else:
            pitch.kdeplot(passes[xcol], passes[ycol], ax=ax,
                          cmap='hot', fill=True, levels=100, alpha=0.7)
        ax.set_title(title, fontsize=14)

    plt.tight_layout()
    plt.show()

In [ ]:
import io, base64, math

EVENTOS_VAEP = [
    'Pass', 'Shot', 'TakeOn', 'Cross', 'Dribble',
    'BallRecovery', 'Clearance', 'Tackle', 'Interception',
    'Dispossessed', 'MissedShots', 'SavedShot', 'Goal'
]

METRICAS_VAEP = [
    ('x',                          'hist', 'Posición X'),
    ('y',                          'hist', 'Posición Y'),
    ('minute',                     'hist', 'Minuto'),
    ('time_since_previous_action', 'hist', 'Tiempo desde acción anterior'),
    ('xT',                         'hist', 'xT'),
    ('outcome_type',               'bar',  'Outcome'),
    ('estado_partido',             'bar',  'Estado del partido'),
]

METRICAS_QUAL = [
    ('Zone',   'bar',  'Zona'),
    ('Length', 'hist', 'Distancia'),
    ('Angle',  'hist', 'Ángulo'),
]

_DESC_EVENTOS = {
    'Pass': 'Pase a un compañero',
    'Shot': 'Tiro al arco',
    'TakeOn': 'Intento de gambeta',
    'Cross': 'Centro',
    'Dribble': 'Gambeta',
    'BallRecovery': 'Recuperación del balón',
    'Clearance': 'Despeje',
    'Tackle': 'Entrada/tackle al rival',
    'Interception': 'Intercepción del balón',
    'Dispossessed': 'Jugador pierde el balón',
    'MissedShots': 'Tiro desviado o al palo',
    'SavedShot': 'Tiro al arco atajado',
    'Goal': 'Gol',
}

def distribucion_vaep(df, df_qual=None, eventos=None, metricas=None, metricas_qual=None):
    if eventos is None:
        eventos = EVENTOS_VAEP
    if metricas is None:
        metricas = METRICAS_VAEP
    if metricas_qual is None:
        metricas_qual = METRICAS_QUAL

    for evento in eventos:
        subset = df[df['event_name'] == evento]
        if len(subset) == 0:
            continue
        n = len(subset)

        plots = []
        for col, kind, title in metricas:
            if col in subset.columns and subset[col].notna().sum() > 0:
                plots.append((col, kind, title, subset[col].dropna()))

        if df_qual is not None:
            sub_qual = df_qual[df_qual.index.isin(subset['id'])]
            for col, kind, title in metricas_qual:
                if col in sub_qual.columns and sub_qual[col].notna().sum() > 0:
                    plots.append((col, kind, title, sub_qual[col].dropna()))

        ncols = 4
        nrows = math.ceil(len(plots) / ncols)

        fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), facecolor='#000000')
        axes = axes.flatten()

        for i, (col, kind, title, data) in enumerate(plots):
            ax = axes[i]
            ax.set_facecolor('#0a0a0a')
            ax.tick_params(colors='#aaa', labelsize=8)
            for spine in ax.spines.values():
                spine.set_edgecolor('#222')
            ax.set_title(title, color='#ddd', fontsize=9)

            if kind == 'hist':
                ax.hist(data, bins=40, color='#48BF5F', edgecolor='none', alpha=0.85)
            else:
                vc = data.value_counts().head(8)
                ax.barh(vc.index[::-1], vc.values[::-1], color='#48BF5F', alpha=0.85)
                ax.tick_params(axis='y', labelsize=7, colors='#aaa')

        for j in range(len(plots), len(axes)):
            axes[j].set_visible(False)

        plt.tight_layout(pad=1.5)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', facecolor='#000000', dpi=120)
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode('utf-8')
        plt.close(fig)

        desc = _DESC_EVENTOS.get(evento, '')
        desc_html = f'<div style="color:#888; font-size:12px; margin-top:3px">{desc}</div>' if desc else ''

        display(HTML(f"""
        <div style="font-family:sans-serif; max-width:1100px; margin-bottom:24px">
          <div style="background:#000; color:white; padding:14px 22px; border-radius:8px 8px 0 0;
                      border-left:5px solid #48BF5F">
            <div style="font-size:18px; font-weight:bold">{evento}
              <span style="font-size:12px; color:#aaa; font-weight:normal; margin-left:10px">{n:,} eventos</span>
            </div>
            {desc_html}
          </div>
          <div style="background:#000; border-radius:0 0 8px 8px; padding:8px">
            <img src="data:image/png;base64,{img_b64}" style="width:100%; border-radius:4px"/>
          </div>
        </div>
        """))